# TBP — Calibración de Ā con la EPH (Opción 2)

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">
DATOS SINTÉTICOS — no son Argentina<br>(las celdas de prueba usan un dataset inventado; sirven solo para verificar el código)
</h1>

**Objetivo.** Estimar Ā (años de aporte promedio por persona de la cohorte, con ceros incluidos) como

Ā_c = Σ_{a=18}^{64} S_c(a) · d_c(a)

donde S_c(a) = P(sobrevivir hasta la edad a) y d_c(a) = P(aportar a la edad a | vivo), aproximada con la **tasa de empleo registrado** de la EPH tratada como pseudo-panel por año de nacimiento (cohorte = ANO4 − CH06) y edad.

**Estado:** este notebook NO produce ningún valor de Ā para Argentina hasta que se coloquen los microdatos reales de la EPH en `datos/eph/` (ver `datos/eph/README.md` y `docs/calibracion_A_EPH.md`). Sin esos archivos termina limpio con el mensaje "faltan datos".

Estructura: (1) configuración y funciones, (2) prueba con datos sintéticos + asserts, (3) prueba del lector de archivos, (4) corrida real (solo si hay datos).

In [ ]:
# ============ 1. CONFIGURACIÓN Y FUNCIONES ============
import os, re, io, glob, zipfile, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EDAD_MIN, EDAD_MAX = 18, 64
EDADES = np.arange(EDAD_MIN, EDAD_MAX + 1)          # 47 edades simples
RUTAS_CANDIDATAS = ["datos/eph", "../datos/eph", "../../datos/eph"]
RUTAS_S65 = ["datos/procesados/tbp_argentina_cohortes_1950_2035.csv",
             "../datos/procesados/tbp_argentina_cohortes_1950_2035.csv"]
MIN_N_CELDA = 20        # mínimo de observaciones (sin ponderar) por celda cohorte x edad
MIN_EDADES_COHORTE = 3  # mínimo de edades observadas para estimar el efecto de una cohorte
GOMPERTZ_B = 0.085      # pendiente de la mortalidad adulta (supuesto; ver docs)
# Regla para cuentapropistas/patrones (la base individual NO trae una variable inequívoca de
# aporte de independientes: a verificar contra el diseño de registro). Opciones:
#   "solo_asalariados": independientes cuentan como NO aportantes (cota inferior)
#   "pp07i":            independientes aportan si PP07I == 1 (uso a verificar; ver docs)
#   "escenario":        independientes aportan con probabilidad IND_SHARE (supuesto explícito)
IND_RULE_DEFAULT = "solo_asalariados"
IND_SHARE_DEFAULT = None
NEEDED = {"ANO4", "TRIMESTRE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"}

def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))

# ---------- (a) lectura de archivos EPH ----------
def find_eph_files(rutas=RUTAS_CANDIDATAS):
    """Busca bases individuales (usu_individual_T*.txt/.csv, o EPH_usu_*.zip) en las rutas candidatas."""
    out = []
    for r in rutas:
        if not os.path.isdir(r):
            continue
        for p in sorted(glob.glob(os.path.join(r, "**", "*"), recursive=True)):
            b = os.path.basename(p).lower()
            if os.path.isfile(p) and re.search(r"usu_individual.*\.(txt|csv)$", b):
                out.append({"path": p, "member": None})
            elif os.path.isfile(p) and b.endswith(".zip") and "eph_usu" in b:
                with zipfile.ZipFile(p) as z:
                    for m in z.namelist():
                        if re.search(r"usu_individual.*\.(txt|csv)$", m.lower()):
                            out.append({"path": p, "member": m})
    return out

def _read_csv_flex(handle_or_path):
    df = pd.read_csv(handle_or_path, sep=";", usecols=lambda c: c.strip().upper() in NEEDED,
                     low_memory=False, encoding="latin-1")
    return df

def read_individual(src):
    """Lee UNA base individual trimestral (solo columnas necesarias). Devuelve DataFrame con columnas en MAYÚSCULAS."""
    if src["member"] is None:
        df = _read_csv_flex(src["path"])
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            df = _read_csv_flex(io.TextIOWrapper(f, encoding="latin-1"))
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = {"ANO4", "CH06", "ESTADO", "PONDERA"} - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {sorted(faltan)} en {src}. Revisar el diseño de registro.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE"):
        if opt not in df.columns:
            df[opt] = np.nan
    return df

def read_all(srcs):
    return pd.concat([read_individual(s) for s in srcs], ignore_index=True)

# ---------- (b) pseudo-panel: densidad por edad x cohorte ----------
def add_flags(df, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None):
    """Filtra 18-64 con entrevista individual realizada y construye 'reg' (prob. de estar aportando).
    Asalariado (CAT_OCUP==3): reg = (PP07H==1). Independiente (1,2): según ind_rule. Resto: 0."""
    d = df[(df["CH06"] >= EDAD_MIN) & (df["CH06"] <= EDAD_MAX) & (df["ESTADO"].isin([1, 2, 3]))].copy()
    if sexo is not None:
        d = d[d["CH04"] == sexo]
    ocupado = d["ESTADO"] == 1
    asal = ocupado & (d["CAT_OCUP"] == 3)
    indep = ocupado & d["CAT_OCUP"].isin([1, 2])
    reg = np.zeros(len(d))
    reg[asal.values] = (d.loc[asal, "PP07H"] == 1).astype(float).values   # Ns/Nr (9) o 2 => no registrado
    if ind_rule == "pp07i":
        reg[indep.values] = (d.loc[indep, "PP07I"] == 1).astype(float).values
    elif ind_rule == "escenario":
        if ind_share is None or not (0 <= ind_share <= 1):
            raise ValueError("ind_rule='escenario' requiere ind_share en [0,1] (supuesto explícito).")
        reg[indep.values] = float(ind_share)
    elif ind_rule != "solo_asalariados":
        raise ValueError(f"ind_rule desconocida: {ind_rule}")
    d["reg"] = reg
    d["edad"] = d["CH06"].astype(int)
    d["cohorte"] = (d["ANO4"] - d["CH06"]).astype(int)     # error de +-1 año (no se usa la fecha exacta)
    d["w"] = d["PONDERA"].astype(float)
    return d[["cohorte", "edad", "w", "reg"]]

def build_cells(d, min_n=MIN_N_CELDA):
    """Densidad ponderada por (cohorte, edad), agregando todos los trimestres/años en que esa cohorte tuvo esa edad."""
    d = d.assign(wreg=d["w"] * d["reg"])
    g = d.groupby(["cohorte", "edad"]).agg(n=("w", "size"), w=("w", "sum"), wreg=("wreg", "sum")).reset_index()
    g["dens"] = g["wreg"] / g["w"]
    g = g[g["n"] >= min_n].reset_index(drop=True)
    return g

# ---------- modelo logit aditivo cohorte + edad (supuesto: perfil por edad estable entre cohortes) ----------
def fit_age_cohort_logit(cells, iters=100, ridge=1e-3):
    coh = np.sort(cells["cohorte"].unique()); eds = np.sort(cells["edad"].unique())
    ci = {c: i for i, c in enumerate(coh)}; ei = {a: i for i, a in enumerate(eds)}
    X = np.zeros((len(cells), len(coh) + len(eds) - 1))
    for r, (c, a) in enumerate(zip(cells["cohorte"], cells["edad"])):
        X[r, ci[c]] = 1.0
        if ei[a] > 0:
            X[r, len(coh) + ei[a] - 1] = 1.0
    y = cells["dens"].values; m = cells["n"].values.astype(float)
    yy = np.clip(y, 0.01, 0.99)
    beta = np.linalg.solve(X.T @ (m[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (m * np.log(yy / (1 - yy))))
    for _ in range(iters):
        eta = X @ beta
        mu = np.clip(1 / (1 + np.exp(-eta)), 1e-6, 1 - 1e-6)
        W = m * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        new = np.linalg.solve(X.T @ (W[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (W * z))
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new; break
        beta = new
    alpha = pd.Series(beta[:len(coh)], index=coh)
    f = pd.Series(np.r_[0.0, beta[len(coh):]], index=eds)
    n_ed = cells.groupby("cohorte")["edad"].nunique().reindex(coh)
    return {"alpha": alpha, "f": f, "n_edades": n_ed, "cells": cells}

def _sig(x): return 1 / (1 + np.exp(-x))

def extrapolar_alpha(fit, cohortes, modo="ultimas_k", k=5, ventana_tendencia=15):
    """Efecto de cohorte para cohortes sin datos suficientes (jóvenes o futuras). Supuesto explícito."""
    ok = fit["alpha"][fit["n_edades"] >= MIN_EDADES_COHORTE]
    out = {}
    for c in cohortes:
        if c in ok.index:
            out[c] = ok[c]
        elif c > ok.index.max():
            if modo == "ultimas_k":
                out[c] = ok.iloc[-k:].mean()
            elif modo == "tendencia":
                w = ok.iloc[-ventana_tendencia:]; b = np.polyfit(w.index.values, w.values, 1)
                out[c] = np.polyval(b, c)
            else:
                raise ValueError(modo)
        else:
            out[c] = ok.iloc[:k].mean()
    return pd.Series(out)

def densidad_modelada(fit, alpha_c):
    """Matriz cohorte x edad (18..64) de d_c(a). Edades sin datos usan el perfil f(a) del modelo."""
    f = fit["f"].reindex(EDADES)
    f = f.interpolate(limit_direction="both")           # edades sin ninguna observación: interpola/extiende el perfil
    return pd.DataFrame(_sig(alpha_c.values[:, None] + f.values[None, :]), index=alpha_c.index, columns=EDADES)

# ---------- (c) supervivencia y Ā ----------
def supervivencia_gompertz(s65, b=GOMPERTZ_B):
    """S(a) desde el nacimiento para a=18..64, Gompertz con pendiente b calibrado para que S(65)=s65 (por cohorte)."""
    s65 = np.asarray(s65, float)
    a = -np.log(s65) * b / (np.exp(65 * b) - 1)
    return np.exp(-(a[:, None] / b) * (np.exp(b * EDADES[None, :]) - 1))

def abar_por_cohorte(dens, s65):
    S = supervivencia_gompertz(s65.reindex(dens.index).values)
    return pd.Series((S * dens.values).sum(axis=1), index=dens.index, name="A_bar"), S

def pipeline(df_ind, s65, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None,
             modo_extrap="ultimas_k", cohortes=None):
    """df individual EPH -> tabla por cohorte. Idéntico para datos sintéticos y reales."""
    d = add_flags(df_ind, ind_rule, ind_share, sexo)
    cells = build_cells(d)
    fit = fit_age_cohort_logit(cells)
    cohortes = list(cohortes) if cohortes is not None else list(s65.index)
    alpha_c = extrapolar_alpha(fit, cohortes, modo=modo_extrap)
    dens = densidad_modelada(fit, alpha_c)
    abar, S = abar_por_cohorte(dens, s65)
    tabla = pd.DataFrame({"A_bar": abar,
                          "n_edades_obs": fit["n_edades"].reindex(abar.index).fillna(0).astype(int),
                          "S65": s65.reindex(abar.index),
                          "A_max_sin_densidad": S.sum(axis=1)})
    return {"tabla": tabla, "dens": dens, "fit": fit, "cells": cells}

def chequeos(res):
    t, dens, cells = res["tabla"], res["dens"], res["cells"]
    assert np.all((cells["dens"] >= 0) & (cells["dens"] <= 1)), "densidad observada fuera de [0,1]"
    assert np.all((dens.values >= 0) & (dens.values <= 1)), "densidad modelada fuera de [0,1]"
    assert np.all((t["A_bar"] >= 0) & (t["A_bar"] <= 47)), "A_bar fuera de [0,47]"
    assert np.all(t["A_bar"] <= t["A_max_sin_densidad"] + 1e-9), "A_bar supera sum S(a) (densidad=1)"
    # monotonía: con la misma densidad, mayor S65 => mayor A_bar
    s_alta = (t["S65"] + 0.02).clip(upper=0.999)
    a2, _ = abar_por_cohorte(dens, s_alta)
    assert np.all(a2.values > t["A_bar"].values - 1e-12), "monotonía en S65 violada"
    return True
print("Funciones cargadas.")

## 2. Prueba con DATASET SINTÉTICO

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">DATOS SINTÉTICOS — no son Argentina</h1>

Se simula una EPH "de juguete" (columnas con los mismos nombres que la base individual de INDEC) con una verdad conocida (efecto de cohorte + perfil por edad + una pequeña oscilación de período), para comprobar que el pipeline recupera Ā. **Los números de esta sección no dicen nada sobre Argentina.**

In [ ]:
# ============ 2. DATASET SINTÉTICO + ASSERTS ============
banner("DATOS SINTÉTICOS — no son Argentina")

def f_edad_sintetica(a):  # perfil por edad inventado (logit)
    return 1.2 * np.exp(-((a - 40) / 18.0) ** 2) - 1.0

def alpha_sintetico(c):   # efecto de cohorte inventado (logit)
    return -0.2 + 0.008 * (np.asarray(c) - 1970)

def periodo_sintetico(y):
    return 0.10 * np.sin((np.asarray(y) - 2003) / 3.0)

def s65_sintetico(c):
    return np.clip(0.70 + 0.0035 * (np.asarray(c) - 1950), 0.5, 0.97)

def generar_eph_sintetica(anios=range(2003, 2025), n_por_edad=25, seed=123):
    rng = np.random.default_rng(seed)
    filas = []
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                p = _sig(alpha_sintetico(y - a) + f_edad_sintetica(a) + periodo_sintetico(y))
                u = rng.random(n_por_edad)
                reg = u < p
                otro = ~reg
                v = rng.random(n_por_edad)
                estado = np.where(reg, 1, np.where(v < 0.45, 1, 3))   # no registrados: 45% ocupados, resto inactivos
                cat = np.where(estado == 1, np.where(reg | (v < 0.30), 3, 2), np.nan)  # parte son cuentapropistas
                pp07h = np.where(cat == 3, np.where(reg, 1, 2), np.nan)
                pp07i = np.where(cat == 2, 2, np.nan)
                filas.append(pd.DataFrame({"ANO4": y, "TRIMESTRE": q, "CH04": rng.integers(1, 3, n_por_edad),
                                           "CH06": a, "ESTADO": estado, "CAT_OCUP": cat, "PP07H": pp07h,
                                           "PP07I": pp07i, "PONDERA": rng.integers(300, 900, n_por_edad)}))
    return pd.concat(filas, ignore_index=True)

df_sint = generar_eph_sintetica()
print("Filas sintéticas:", len(df_sint))
cohortes_sint = np.arange(1940, 2036)
s65_sint = pd.Series(s65_sintetico(cohortes_sint), index=cohortes_sint)
res_s = pipeline(df_sint, s65_sint, cohortes=cohortes_sint)
t = res_s["tabla"]

# verdad conocida
dens_true = pd.DataFrame(_sig(alpha_sintetico(cohortes_sint)[:, None] + f_edad_sintetica(EDADES)[None, :]),
                         index=cohortes_sint, columns=EDADES)
a_true, _ = abar_por_cohorte(dens_true, s65_sint)
t["A_bar_verdad_sintetica"] = a_true
t["error"] = t["A_bar"] - t["A_bar_verdad_sintetica"]

# --- asserts de control ---
chequeos(res_s)
obs = t[t["n_edades_obs"] >= MIN_EDADES_COHORTE]
mae = obs["error"].abs().mean()
print(f"[sintético] error absoluto medio de Ā en cohortes observadas: {mae:.3f} años")
assert mae < 1.5, "el pipeline no recupera la verdad sintética (tolerancia 1,5 años; el efecto de período sintético se confunde parcialmente con cohorte/edad: problema APC)"
fut = t[t["n_edades_obs"] < MIN_EDADES_COHORTE]
print(f"[sintético] error medio en cohortes SIN datos (extrapoladas con 'ultimas_k'): {fut['error'].mean():+.3f} años "
      "(sesgo esperado si hay tendencia entre cohortes: es el costo del supuesto de extrapolación)")
# monotonía en S65 a igual densidad (test directo)
a_lo, _ = abar_por_cohorte(res_s["dens"], s65_sint - 0.05)
a_hi, _ = abar_por_cohorte(res_s["dens"], s65_sint + 0.02)
assert np.all(a_hi.values > a_lo.values), "Ā debería crecer con S65"
# densidad cero => Ā cero ; densidad 1 => Ā = sum S(a) <= 47
z, _ = abar_por_cohorte(res_s["dens"] * 0.0, s65_sint); assert np.allclose(z, 0)
u, S1 = abar_por_cohorte(res_s["dens"] * 0.0 + 1.0, s65_sint); assert np.all(u <= 47) and np.all(u > 0)
print("Todos los asserts sintéticos OK.")

# gráficos
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for c, col in [(1950, "C0"), (1970, "C1"), (1990, "C2"), (2000, "C3")]:
    cc = res_s["cells"][res_s["cells"]["cohorte"] == c]
    ax[0].scatter(cc["edad"], cc["dens"], s=10, color=col, alpha=.6)
    ax[0].plot(EDADES, res_s["dens"].loc[c].values, color=col, label=f"cohorte {c}")
ax[0].set_xlabel("edad"); ax[0].set_ylabel("densidad d(a)"); ax[0].legend(); ax[0].set_title("SINTÉTICO: puntos observados vs modelo")
ax[1].plot(t.index, t["A_bar"], label="estimado"); ax[1].plot(t.index, t["A_bar_verdad_sintetica"], "--", label="verdad sintética")
ax[1].set_xlabel("cohorte"); ax[1].set_ylabel("Ā (años)"); ax[1].legend()
ax[1].set_title("SINTÉTICO: Ā por cohorte")
fig.suptitle("DATOS SINTÉTICOS — no son Argentina", color="red", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()
banner("FIN PRUEBA SINTÉTICA — estos números NO son Argentina")

## 3. Prueba del lector de archivos (con un archivo de juguete en formato INDEC)

Escribe un subconjunto del dataset **sintético** en una carpeta temporal con el nombre `usu_individual_T100.txt` (separador `;`), y verifica que `find_eph_files` + `read_all` + `pipeline` dan lo mismo que con el DataFrame en memoria. No toca `datos/eph/`.

In [ ]:
banner("DATOS SINTÉTICOS — prueba del lector, no son Argentina")
with tempfile.TemporaryDirectory() as tmp:
    sub = df_sint[df_sint["ANO4"].isin([2010, 2011, 2012, 2013])].copy()
    sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]  # mezcla de mayúsculas/minúsculas
    sub.to_csv(os.path.join(tmp, "usu_individual_T100.txt"), sep=";", index=False)
    srcs = find_eph_files([tmp]); assert len(srcs) == 1
    leido = read_all(srcs)
    assert set(NEEDED) <= set(leido.columns)
    assert len(leido) == len(sub)
    # zip con el nombre típico de INDEC
    zp = os.path.join(tmp, "EPH_usu_1_Trim_2012_txt.zip")
    with zipfile.ZipFile(zp, "w") as z:
        z.write(os.path.join(tmp, "usu_individual_T100.txt"), "usu_individual_T112.txt")
    os.remove(os.path.join(tmp, "usu_individual_T100.txt"))
    srcs2 = find_eph_files([tmp]); assert len(srcs2) == 1 and srcs2[0]["member"] is not None
    assert len(read_all(srcs2)) == len(sub)
print("Lector OK (txt y zip) con datos sintéticos.")
# find_eph_files sobre una carpeta inexistente => lista vacía, sin error
assert find_eph_files(["/ruta/que/no/existe"]) == []

## 4. Corrida con datos REALES (solo si están en `datos/eph/`)

Si no hay archivos, el notebook termina limpio con "faltan datos". Si los hay, aplica el mismo `pipeline` y guarda `salidas_tbp/abar_eph_por_cohorte.csv` (carpeta ignorada por git). Las cifras de referencia 14,2 (ANSES, observado, sesgado a la baja) y 30 (requisito legal de referencia, no un máximo) se muestran solo como contexto; **no son objetivo de calibración**.

In [ ]:
# ============ 4. CORRIDA REAL ============
srcs_reales = find_eph_files(RUTAS_CANDIDATAS)
HAY_DATOS = len(srcs_reales) > 0

if not HAY_DATOS:
    print("=" * 78)
    print("FALTAN DATOS: no se encontraron bases individuales de la EPH en", RUTAS_CANDIDATAS)
    print("Qué hacer: bajar las bases individuales (usu_individual_T*.txt o EPH_usu_*_txt.zip) desde INDEC")
    print("y dejarlas en datos/eph/ (ver datos/eph/README.md). No se calculó ningún Ā para Argentina.")
    print("=" * 78)
else:
    s65_path = next((p for p in RUTAS_S65 if os.path.exists(p)), None)
    if s65_path is None:
        print("FALTAN DATOS: no se encontró tbp_argentina_cohortes_1950_2035.csv (S65 por cohorte).")
        HAY_DATOS = False
if HAY_DATOS:
    print(f"Archivos EPH encontrados: {len(srcs_reales)}")
    df_real = read_all(srcs_reales)
    print("Años cubiertos:", int(df_real["ANO4"].min()), "-", int(df_real["ANO4"].max()), "| filas:", len(df_real))
    s65_real = pd.read_csv(s65_path).set_index("cohorte")["S65"]
    coh_real = s65_real.index
    escenarios = {"solo_asalariados": dict(ind_rule="solo_asalariados"),
                  "pp07i": dict(ind_rule="pp07i"),
                  "indep_50pct (supuesto)": dict(ind_rule="escenario", ind_share=0.5)}
    salida = {}
    for nombre, kw in escenarios.items():
        r = pipeline(df_real, s65_real, cohortes=coh_real, **kw)
        chequeos(r)
        salida[nombre] = r["tabla"]["A_bar"]
        salida["n_edades_obs"] = r["tabla"]["n_edades_obs"]
    out = pd.DataFrame(salida)
    os.makedirs("salidas_tbp", exist_ok=True)
    out.to_csv("salidas_tbp/abar_eph_por_cohorte.csv")
    print(out.loc[[c for c in (1960, 1970, 1980, 1990, 2000) if c in out.index]].round(2))
    print("Referencias de contexto (NO objetivo): ANSES observado 14,2 (sesgado a la baja); requisito legal 30 (techo).")